## Neuro Representation Learning with AraBERT

A pretrained transformer encoder (AraBERT) is employed to capture
deep contextual semantic representations of Arabic sequences.

The model pipeline includes:

1. Transformer sequence encoding
2. Extraction of the [CLS] contextual embedding
3. Dropout regularization layer
4. Sigmoid classification head for binary simile prediction

Binary cross-entropy loss and Adam optimization are used for training.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding
from arabert.preprocess import ArabertPreprocessor
from transformers import TFAutoModel
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
from symbolic_model import build_symbolic_matrix


In [ ]:
ModelName = "aubmindlab/bert-large-arabertv02"
MAX_LEN = 120
BATCH_SIZE = 16
EPOCHS = 2
SEED = 42


tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
import pandas as pd

df = pd.read_csv("dataset.csv")

sentences = df["sentence"].tolist()
labels = df["label"].tolist()

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(sentences, labels, test_size=0.1, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=SEED)
X_train_sym = build_symbolic_matrix(X_train)
X_val_sym = build_symbolic_matrix(X_val)
X_test_sym = build_symbolic_matrix(X_test)
arabert_prep = ArabertPreprocessor(model_name=ModelName)
tokenizer = AutoTokenizer.from_pretrained(ModelName)

def PreprocessTexts(texts):
    texts = [arabert_prep.preprocess(text) for text in texts]
    encodings = tokenizer(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )
    return encodings['input_ids'], encodings['attention_mask']

X_train_ids, X_train_mask =PreprocessTexts(X_train)
X_val_ids, X_val_mask =PreprocessTexts(X_val)
X_test_ids, X_test_mask =PreprocessTexts(X_test)

In [ ]:
def create_arabert_model(model_name, max_len):

    word_inputs = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    attention_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")
    sym_input = tf.keras.Input(shape=(8,), dtype=tf.float32, name="symbolic_features")

    arabert = TFAutoModel.from_pretrained(model_name, from_pt=True)
    outputs = arabert(word_inputs, attention_mask=attention_mask)[0]

    cls = outputs[:, 0, :]

    combined = tf.keras.layers.Concatenate()([cls, sym_input])

    x = tf.keras.layers.Dense(256, activation='relu')(combined)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(
        inputs=[word_inputs, attention_mask, sym_input],
        outputs=output
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(2e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model = create_arabert_model(ModelName, MAX_LEN)

history = model.fit(
    {
        "input_ids": X_train_ids,
        "attention_mask": X_train_mask,
        "symbolic_features": X_train_sym
    },
    np.array(y_train),
    validation_data=(
        {
            "input_ids": X_val_ids,
            "attention_mask": X_val_mask,
            "symbolic_features": X_val_sym
        },
        np.array(y_val)
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


In [ ]:
y_pred_prob = model.predict({
    "input_ids": X_test_ids,
    "attention_mask": X_test_mask,
    "symbolic_features": X_test_sym
})

y_pred = (y_pred_prob > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
n_bootstraps = 100
accs = []
y_test = np.array(y_test)

for _ in range(n_bootstraps):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    acc = np.mean(y_test[idx] == y_pred[idx])
    accs.append(acc)

mean_acc = np.mean(accs)
std_acc = np.std(accs)

plt.figure(figsize=(4,6))
plt.bar(['Accuracy'], [mean_acc], yerr=[std_acc], capsize=10)
plt.ylim(0, 1)
plt.title("Test Accuracy ± Error Bar")
plt.show()